# Drift–skill relationship figures

This plotting-only notebook focuses on the drift–skill relationship formerly shown in Section 6 of the combined analysis notebook. It reads only the multi-region source products written by `0_run_drift_diag.ipynb`. Separate figures are produced for Niño3.4 and the North Atlantic, with global land retained for H2OSOI.

Each point is one initialization year. The horizontal coordinate is the mean early change in absolute distance to observations, and the vertical coordinate is the RMS observation error in a later lead window. Negative early drift means movement closer to observations; positive early drift means movement away.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import pandas as pd
import xarray as xr

from esp_lab.diagnostics.two_reference_drift import (
    compute_early_drift_late_error,
)

### Supporting configuration

In [ ]:
OUTPUT_ROOT = Path(
    '/global/cfs/cdirs/e3sm/S2S2D/s2d_diag/multimodel/leadtime_drift/'
    'two_reference'
)
FIGURE_ROOT = Path('/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag')
VARIABLES = ('TREFHT', 'SST', 'PSL', 'PRECT', 'H2OSOI')
INIT_MONTHS = (5, 11)
SOURCES = ('JRA55_FOSIRL', 'Reanalysis')
SOURCE_LABELS = {'JRA55_FOSIRL': 'FOSIRL', 'Reanalysis': 'Reanalysis'}
SOURCE_COLORS = {'JRA55_FOSIRL': 'tab:blue', 'Reanalysis': 'tab:orange'}
SOURCE_MARKERS = {'JRA55_FOSIRL': 'o', 'Reanalysis': 's'}
MONTH_NAMES = {5: 'May', 11: 'November'}
PLOT_REGIONS = {
    'TREFHT': ('Nino3.4', 'North_Atlantic'),
    'SST': ('Nino3.4', 'North_Atlantic'),
    'PSL': ('Nino3.4', 'North_Atlantic'),
    'PRECT': ('Nino3.4', 'North_Atlantic'),
    'H2OSOI': ('Global_land',),
}
REGION_LABELS = {
    'Nino3.4': 'Niño3.4 (5°S–5°N, 190°–240°E)',
    'North_Atlantic': 'North Atlantic (0°–60°N, 80°W–0°)',
    'Global_land': 'Global land',
}
REGION_FILE_LABELS = {
    'Nino3.4': 'Nino3_4',
    'North_Atlantic': 'North_Atlantic',
    'Global_land': 'Global_land',
}
EARLY_LEADS = (1, 2, 3)
LATE_ERROR_WINDOWS = tuple(
    tuple(range(start, start + 3)) for start in range(7, 25, 3)
)
FIGURE_DPI = 180
FIGURE_SCALE = 1.0
FIGURE_SIZE = (14, 19)
SHOW_FIGURES_INLINE = True
FIGURE_PREFIX = 'fig_leadtime_drift_two_reference'
FONT_SIZE = 11
TITLE_FONT_SIZE = 14
LEGEND_FONT_SIZE = 9
ANNOTATION_FONT_SIZE = 9
SCATTER_SIZE = 34
SCATTER_ALPHA = 0.75

FIGURE_ROOT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({
    'font.size': FONT_SIZE,
    'axes.titlesize': FONT_SIZE,
    'axes.labelsize': FONT_SIZE,
    'figure.titlesize': TITLE_FONT_SIZE,
    'legend.fontsize': LEGEND_FONT_SIZE,
})


def figure_size():
    return tuple(FIGURE_SCALE * value for value in FIGURE_SIZE)


def lead_window_label(leads):
    return f'L{int(leads[0])}–L{int(leads[-1])}'

### Load the required regional fields

Only `delta_abs_e_obs` and `e_obs` are needed. Spatial maps, skill metrics, regime fractions, and paired products are not opened.

In [ ]:
manifest_path = OUTPUT_ROOT / 'two_reference_drift_figure_data_manifest.csv'
if not manifest_path.is_file():
    raise FileNotFoundError(
        f'Missing {manifest_path}; run jupyter/0_run_drift_diag.ipynb first'
    )
full_manifest = pd.read_csv(manifest_path)
regional_manifest = full_manifest.loc[
    full_manifest['variable'].isin(VARIABLES)
    & full_manifest['init_month'].isin(INIT_MONTHS)
    & full_manifest['source'].isin(SOURCES)
    & full_manifest['product'].eq('regional')
].copy()
expected_products = len(VARIABLES) * len(INIT_MONTHS) * len(SOURCES)
if len(regional_manifest) != expected_products:
    raise ValueError(
        f'Expected {expected_products} regional products, found '
        f'{len(regional_manifest)}'
    )
if regional_manifest.duplicated(['variable', 'init_month', 'source']).any():
    raise ValueError('Regional manifest contains duplicate products')


def regional_product_path(variable, init_month, source):
    match = regional_manifest.loc[
        regional_manifest['variable'].eq(variable)
        & regional_manifest['init_month'].eq(init_month)
        & regional_manifest['source'].eq(source),
        'path',
    ]
    if len(match) != 1:
        raise ValueError(
            f'Expected one product for {variable}, init={init_month:02d}, '
            f'{source}; found {len(match)}'
        )
    return Path(match.iloc[0])


def load_drift_skill_inputs(variable, init_month, source):
    path = regional_product_path(variable, init_month, source)
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f'Missing regional product: {path}')
    names = ('delta_abs_e_obs', 'e_obs')
    with xr.open_dataset(path) as opened:
        missing = set(names) - set(opened.data_vars)
        if missing:
            raise KeyError(f'{path} is missing fields: {sorted(missing)}')
        selected = opened[list(names)].load()
    if 'region' not in selected.dims:
        raise ValueError(
            f'{path} predates multi-region output; rerun '
            'jupyter/0_run_drift_diag.ipynb'
        )
    missing_regions = set(PLOT_REGIONS[variable]) - set(
        selected.region.values.astype(str)
    )
    if missing_regions:
        raise ValueError(
            f'{path} is missing regions {sorted(missing_regions)}; rerun '
            'jupyter/0_run_drift_diag.ipynb'
        )
    return selected


drift_skill_inputs = {
    (variable, init_month, source): load_drift_skill_inputs(
        variable, init_month, source
    )
    for variable in VARIABLES
    for init_month in INIT_MONTHS
    for source in SOURCES
}
display(regional_manifest.sort_values(['variable', 'init_month', 'source']))
print(f'Loaded {len(drift_skill_inputs)} regional drift–skill inputs.')

## 6. Drift–skill relationship figures

For initialization year $k$, early drift is

$$D_{\mathrm{early}}(k)=\operatorname{mean}_{L=1,2,3}\Delta|e_{\mathrm{obs}}|(k,L),$$

and later error is the RMS observation departure over the lead window shown in each panel. The annotation reports the Pearson correlation across initialization years. A positive correlation means that starts moving farther from observations early tend to have larger later errors.

In [ ]:
def finite_xy(relationship):
    x = np.asarray(relationship.early_drift).reshape(-1)
    y = np.asarray(relationship.late_error).reshape(-1)
    finite = np.isfinite(x) & np.isfinite(y)
    return x[finite], y[finite]


def add_regression_line(ax, x, y, color):
    if x.size < 2 or np.allclose(x, x[0]):
        return
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(float(x.min()), float(x.max()), 100)
    ax.plot(x_line, slope * x_line + intercept, color=color, linewidth=1.5)


figure_paths = []
correlation_rows = []
for variable in VARIABLES:
    for region_name in PLOT_REGIONS[variable]:
        fig, axes = plt.subplots(
            len(LATE_ERROR_WINDOWS), len(INIT_MONTHS),
            figsize=figure_size(), sharex=True, squeeze=False,
            constrained_layout=True,
        )
        for row, late_leads in enumerate(LATE_ERROR_WINDOWS):
            for col, init_month in enumerate(INIT_MONTHS):
                ax = axes[row, col]
                annotation_lines = []
                for source in SOURCES:
                    fields = drift_skill_inputs[(
                        variable, init_month, source
                    )].sel(region=region_name)
                    relationship = compute_early_drift_late_error(
                        fields.delta_abs_e_obs, fields.e_obs,
                        early_leads=EARLY_LEADS, late_leads=late_leads,
                    )
                    x, y = finite_xy(relationship)
                    correlation = float(relationship.correlation)
                    ax.scatter(
                        x, y, s=SCATTER_SIZE, alpha=SCATTER_ALPHA,
                        marker=SOURCE_MARKERS[source],
                        color=SOURCE_COLORS[source],
                        edgecolor='white', linewidth=0.5,
                        label=SOURCE_LABELS[source],
                    )
                    add_regression_line(ax, x, y, SOURCE_COLORS[source])
                    annotation_lines.append(
                        (SOURCE_LABELS[source], correlation, x.size, source)
                    )
                    correlation_rows.append({
                        'variable': variable, 'region': region_name,
                        'init_month': init_month, 'source': source,
                        'early_leads': lead_window_label(EARLY_LEADS),
                        'late_leads': lead_window_label(late_leads),
                        'correlation': correlation, 'n_starts': x.size,
                    })
                ax.axvline(0, color='0.35', linewidth=0.8, linestyle='--')
                for line_index, (label, correlation, count, source) in enumerate(
                    annotation_lines
                ):
                    ax.text(
                        0.02, 0.97 - 0.10 * line_index,
                        f'{label}: r={correlation:+.2f}, n={count}',
                        transform=ax.transAxes, ha='left', va='top',
                        color=SOURCE_COLORS[source],
                        fontsize=ANNOTATION_FONT_SIZE,
                    )
                ax.set_title(
                    f'{MONTH_NAMES[init_month]} initialization · '
                    f'late error {lead_window_label(late_leads)}'
                )
                ax.set_ylabel('Later RMS observation error')
                ax.grid(True, alpha=0.25)
                if row == 0 and col == len(INIT_MONTHS) - 1:
                    ax.legend(frameon=False)
                if row == len(LATE_ERROR_WINDOWS) - 1:
                    ax.set_xlabel(
                        f'Early mean Δ|e_obs|, '
                        f'{lead_window_label(EARLY_LEADS)} '
                        '(negative = closer; positive = farther)'
                    )
        fig.suptitle(
            f'{variable}: early observation-relative drift versus later error, '
            f'{REGION_LABELS[region_name]}'
        )
        path = FIGURE_ROOT / (
            f'{FIGURE_PREFIX}_{variable}_May_Nov_'
            f'{REGION_FILE_LABELS[region_name]}_drift_skill.png'
        )
        fig.savefig(path, dpi=FIGURE_DPI, bbox_inches='tight')
        if SHOW_FIGURES_INLINE:
            display(fig)
        plt.close(fig)
        figure_paths.append(path)

correlation_table = pd.DataFrame(correlation_rows)
correlation_path = FIGURE_ROOT / (
    f'{FIGURE_PREFIX}_drift_skill_correlations.csv'
)
correlation_table.to_csv(correlation_path, index=False)
display(correlation_table.round({'correlation': 3}))
print(correlation_path)

### Output validation

In [ ]:
expected_figure_count = sum(
    len(PLOT_REGIONS[variable]) for variable in VARIABLES
)
expected_rows = (
    expected_figure_count * len(INIT_MONTHS)
    * len(LATE_ERROR_WINDOWS) * len(SOURCES)
)
assert len(figure_paths) == expected_figure_count
assert all(path.is_file() and path.stat().st_size > 0 for path in figure_paths)
assert len(correlation_table) == expected_rows
assert correlation_path.is_file() and correlation_path.stat().st_size > 0
assert correlation_table['n_starts'].ge(2).all()
assert correlation_table['correlation'].dropna().between(-1, 1).all()
print(
    f'Validated {len(figure_paths)} drift–skill figures and '
    f'{len(correlation_table)} correlation records.'
)